# Imports and paths

In [2]:
# imports and paths
from pathlib import Path
from cosipy.response import RspConverter
import numpy as np
import gzip

response_dir = Path(".")
# only support nonsparse rsp as of 2026-08-27
rsp_file_name = "Imaging_HealPixO3_10ebins.p1.binnedimaging.imagingresponse.nonsparse.rsp.gz"
rsp_file_path = response_dir / rsp_file_name


h5_file_name = "Imaging_HealPixO3_10ebins.p1.binnedimaging.imagingresponse.nonsparse.h5"
h5_file_path = response_dir / h5_file_name

# Inspect the format quikly

In [5]:
import gzip
from pathlib import Path


def inspect_rsp_format(rsp_path):
    rsp_path = Path(rsp_path)

    opener = gzip.open if rsp_path.suffix == ".gz" else open

    matrix_sparse = None
    data_marker = None
    data_size = None

    with opener(rsp_path, "rt") as file:
        for line in file:
            fields = line.split()

            if not fields:
                continue

            if fields[0] == "MS":
                matrix_sparse = fields[1].lower() == "true"

            elif fields[0] == "StartStream":
                data_marker = "StartStream"
                data_size = int(fields[1])
                break

            elif fields[0] == "RD":
                data_marker = "RD"
                break

    print("File:", rsp_path.name)
    print("MS:", matrix_sparse)
    print("Data marker:", data_marker)
    print("Number of dense bins:", data_size)
    print(
        "Supported by RspConverter:",
        matrix_sparse is False and data_marker == "StartStream",
    )
    

inspect_rsp_format(rsp_file_name)

File: Imaging_HealPixO3_10ebins.p1.binnedimaging.imagingresponse.nonsparse.rsp.gz
MS: False
Data marker: StartStream
Number of dense bins: 1769472000
Supported by RspConverter: True


# Determine the count data type

To avoid repeating the dtype-detection scan in future conversions, first determine the smallest integer data type capable of storing the count in every response bin.

If the required dtype is already known for this exact RSP file, this step can be skipped. Otherwise, run the probe below and record the resulting dtype for future conversions.

> Note: The initial probe still reads the entire RSP file once. The first probe-plus-conversion workflow therefore reads the file twice in total. Subsequent conversions can skip the probe and read the RSP only once by explicitly passing the recorded dtype as `elt_type`.

In [17]:
converter = RspConverter(
    norm="Linear",
    norm_params=[10, 10000],
    quiet=False,              # False 表示启用进度条
    bufsize=50_000_000,
)

dtype = converter._get_min_elt_type(rsp_file_path)

print("Required dtype:", dtype)

Getting type for counts:   0%|          | 0/1769472000 [00:00<?, ?it/s]

Required dtype: uint16


# Estimate the RAM required for RSP conversion from the response header

In [7]:
# Set this to the dtype obtained from the probe, e.g. np.uint8.
# Use None if the required dtype is still unknown.
known_dtype = np.uint8

# Use the same buffer size planned for RspConverter.
bufsize = 50_000_000

# A heuristic allowance for Python, NumPy, parsing, HDF5, and compression
# buffers. The converter does not require two full copies of the response.
planning_factor = 1.25


def read_rsp_header(rsp_path):
    """Read the RSP header and return its text and total number of bins."""

    rsp_path = Path(rsp_path)
    opener = gzip.open if rsp_path.suffix.lower() == ".gz" else open

    header_lines = []
    nbins = None

    with opener(rsp_path, "rt") as file:
        for line in file:
            header_lines.append(line.rstrip())

            fields = line.split()
            if fields and fields[0] == "StartStream":
                nbins = int(fields[1])
                break

    if nbins is None:
        raise ValueError("Could not find 'StartStream' in the RSP header.")

    return "\n".join(header_lines), nbins


def to_gib(nbytes):
    """Convert bytes to GiB."""

    return nbytes / 2**30


header, nbins = read_rsp_header(rsp_file_path)

print(f"RSP file: {rsp_file_path}")
print(f"Number of response bins: {nbins:,}")
print()

if known_dtype is not None:
    # Case 1: the dtype is already known
    dtype = np.dtype(known_dtype)
    counts_bytes = nbins * dtype.itemsize
    planning_bytes = counts_bytes * planning_factor

    print("Known-dtype estimate")
    print("--------------------")
    print(f"Count dtype: {dtype.name}")
    print(f"Bytes per bin: {dtype.itemsize}")
    print(f"Theoretical counts-array RAM: {to_gib(counts_bytes):.2f} GiB")
    print(
        f"Planning estimate with {planning_factor:.0%} allowance: "
        f"{to_gib(planning_bytes):.2f} GiB"
    )
    print()
    print(
        "Round this estimate up to the next memory tier available in "
        "the Jupyter Interactive App."
    )

else:
    # Case 2: the dtype is unknown
    print("Unknown-dtype estimate")
    print("----------------------")
    print(
        "The RSP header contains the number of bins, but not the maximum "
        "count. Therefore, the required dtype cannot be determined from "
        "the header alone."
    )
    print()
    print(f"{'Possible dtype':<16}{'Counts RAM':>15}{'Planning estimate':>22}")
    print("-" * 53)

    for candidate in (np.uint8, np.uint16, np.uint32, np.uint64):
        dtype = np.dtype(candidate)
        counts_bytes = nbins * dtype.itemsize
        planning_bytes = counts_bytes * planning_factor

        print(
            f"{dtype.name:<16}"
            f"{to_gib(counts_bytes):>12.2f} GiB"
            f"{to_gib(planning_bytes):>19.2f} GiB"
        )

    # During the dtype probe, only bounded text and uint64 parsing buffers
    # are retained. This is a conservative buffer-only estimate and does
    # not include the Python environment itself.
    probe_buffer_bytes = 10 * bufsize

    print()
    print(
        "Approximate upper estimate for dtype-probe working buffers: "
        f"{to_gib(probe_buffer_bytes):.2f} GiB"
    )
    print(
        "A 4–8 GiB allocation is normally sufficient for the dtype probe; "
        "the full conversion requires the RAM shown in the table."
    )

RSP file: Imaging_HealPixO3_10ebins.p1.binnedimaging.imagingresponse.nonsparse.rsp.gz
Number of response bins: 1,769,472,000

Known-dtype estimate
--------------------
Count dtype: uint8
Bytes per bin: 1
Theoretical counts-array RAM: 1.65 GiB
Planning estimate with 125% allowance: 2.06 GiB

Round this estimate up to the next memory tier available in the Jupyter Interactive App.


# Check if you need to define the spectrum params for conversion

In [13]:
import gzip
from pathlib import Path


def report_rsp_spectrum(rsp_path, converter):
    """
    Report which spectrum normalization will be used during conversion.

    This function only reads the RSP header. It does not read the
    response counts into memory.
    """

    rsp_path = Path(rsp_path)
    opener = gzip.open if rsp_path.suffix == ".gz" else open

    header_spectrum = None
    invalid_spectrum = None

    with opener(rsp_path, "rt") as file:
        for raw_line in file:
            fields = raw_line.split()

            if not fields or fields[0].startswith("#"):
                continue

            if fields[0] == "SP":
                # An empty SP line is treated as missing by RspConverter
                if len(fields) == 1:
                    continue

                norm = fields[1]
                raw_params = fields[2:]

                try:
                    # Use the converter's own validation rules
                    parsed_params = converter._validate_norm_params(
                        norm,
                        raw_params,
                    )
                except (TypeError, ValueError) as error:
                    invalid_spectrum = {
                        "line": raw_line.strip(),
                        "error": str(error),
                    }
                    break

                header_spectrum = {
                    "line": raw_line.strip(),
                    "norm": norm,
                    "params": parsed_params,
                }

            # Stop before reading the response data
            elif fields[0] in {"StartStream", "RD"}:
                break

    print(f"RSP file: {rsp_path.name}")

    if invalid_spectrum is not None:
        print("SP status: present but invalid")
        print(f"Header entry: {invalid_spectrum['line']}")
        print(f"Validation error: {invalid_spectrum['error']}")
        print("Conversion behavior: conversion will fail")
        print(
            "Fallback behavior: converter.norm and converter.norm_params "
            "will NOT be used automatically"
        )

    elif header_spectrum is not None:
        print("SP status: complete and usable")
        print(f"Header entry: {header_spectrum['line']}")
        print(f"Spectrum source: RSP header")
        print(f"Normalization: {header_spectrum['norm']}")
        print(f"Normalization parameters: {header_spectrum['params']}")
        print(
            "Fallback behavior: converter.norm and converter.norm_params "
            "will NOT take effect"
        )

    else:
        print("SP status: missing or empty")
        print("Spectrum source: RspConverter fallback configuration")

        try:
            fallback_params = converter._validate_norm_params(
                converter.norm,
                converter.norm_params,
            )

            print(f"Normalization: {converter.norm}")
            print(f"Normalization parameters: {fallback_params}")
            print(
                "Fallback behavior: converter.norm and "
                "converter.norm_params WILL take effect"
            )

        except (TypeError, ValueError) as error:
            print("Fallback status: invalid")
            print(f"Validation error: {error}")
            print("Conversion behavior: conversion will fail")

In [14]:
converter = RspConverter(
    quiet=False,
    bufsize=50_000_000,
)

report_rsp_spectrum(
    rsp_file_path,
    converter,
)

RSP file: Imaging_HealPixO3_10ebins.p1.binnedimaging.imagingresponse.nonsparse.rsp.gz
SP status: complete and usable
Header entry: SP Linear 10 10000
Spectrum source: RSP header
Normalization: Linear
Normalization parameters: (10, 10000)
Fallback behavior: converter.norm and converter.norm_params will NOT take effect


# Convert the response

In [20]:
converter = RspConverter(
    norm="Linear",               # 只在 RSP 缺少 SP 时使用
    norm_params=[10, 10000],     # 只在 RSP 缺少 SP 时使用
    quiet=False,
    bufsize=50_000_000,
)

output = converter.convert_to_h5(
    rsp_filename=rsp_file_path,
    h5_filename=h5_file_path,
    overwrite=False,
    compress=True,
    elt_type=np.uint16,  # 使用已经探测到的结果
    pa_convention=None,
)

print(output)

Reading counts:   0%|          | 0/1769472000 [00:00<?, ?it/s]

Writing chunks:   0%|          | 0/7680 [00:00<?, ?it/s]

Imaging_HealPixO3_10ebins.p1.binnedimaging.imagingresponse.nonsparse.h5


# Valid and check axes

In [21]:
import h5py as h5
import hdf5plugin

with h5.File(h5_file_path, "r") as f:
    counts = f["DRM/COUNTS"]
    properties = counts.id.get_create_plist()

    print("Number of filters:", properties.get_nfilters())

    for index in range(properties.get_nfilters()):
        filter_id, flags, parameters, name = properties.get_filter(index)

        if isinstance(name, bytes):
            name = name.decode(errors="replace")

        print("Filter ID:", filter_id)
        print("Filter name:", name)
        print("Filter parameters:", parameters)

Number of filters: 1
Filter ID: 32008
Filter name: bitshuffle; see https://github.com/kiyo-masui/bitshuffle
Filter parameters: (0, 4, 2, 0, 2)


In [22]:
from cosipy.response import FullDetectorResponse

with FullDetectorResponse.open(h5_file_path) as response:
    print("Response shape:", response.shape)
    print("Axis labels:", response.axes.labels)
    print("Response unit:", response.unit)

    # 读取较小的测试切片，不加载完整 response
    sample_counts = response.get_counts(
        0,
        em_slice=slice(0, 1),
    )

    print("Sample counts shape:", sample_counts.shape)
    print("Sample counts dtype:", sample_counts.dtype)

Response shape: (768, 10, 10, 30, 768)
Axis labels: ['NuLambda' 'Ei' 'Em' 'Phi' 'PsiChi']
Response unit: cm2
Sample counts shape: (10, 1, 30, 768)
Sample counts dtype: uint16


# Check Compression Level

In [23]:
from pathlib import Path

import h5py as h5
import hdf5plugin

h5_path = Path(h5_file_path)

with h5.File(h5_path, "r") as f:
    counts = f["DRM/COUNTS"]

    logical_bytes = counts.size * counts.dtype.itemsize
    physical_bytes = h5_path.stat().st_size

    print(f"Logical COUNTS size: {logical_bytes / 2**30:.3f} GiB")
    print(f"Physical HDF5 size: {physical_bytes / 2**30:.3f} GiB")
    print(f"Compression ratio: {logical_bytes / physical_bytes:.2f}x")
    print(
        "Reduction relative to float32 representation: "
        f"{logical_bytes * 4 / physical_bytes:.2f}x"
    )

Logical COUNTS size: 3.296 GiB
Physical HDF5 size: 0.055 GiB
Compression ratio: 60.34x
Reduction relative to float32 representation: 241.34x
